# Tool-Router Ladder: teacher-forced generation (zero-shot and SFT)

**Settings:** GPU T4, Internet on, the dataset attached, and secret `HF_TOKEN` ticked.

The Mac renders every prompt and scores every completion. This notebook only maps prompts to completions.

**Download afterwards:** each `completions-*.jsonl` and its `.meta.json` from `/kaggle/working/out/`. Put them in `results/` locally.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN"])

In [ ]:
!pip install -q vllm

# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "madhusiddharths1"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

OUT = '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)

### Zero-shot (for Gate 2)

The third run is the review's handicap check: the same 1.5B model, but with tools rendered the way Qwen's own template does.

In [ ]:
!python src/generate.py --config configs/qwen05b.yaml --prompts "$DATA/prompts-c16.jsonl" --out "$OUT/completions-qwen05b-zeroshot-c16.jsonl"
!python src/generate.py --config configs/qwen15b.yaml --prompts "$DATA/prompts-c16.jsonl" --out "$OUT/completions-qwen15b-zeroshot-c16.jsonl"
!python src/generate.py --config configs/qwen15b.yaml --prompts "$DATA/prompts-c16-json.jsonl" --out "$OUT/completions-qwen15b-zeroshot-c16-json.jsonl"

### Fine-tuned students (after the training notebook has pushed both adapters)

In [ ]:
!python src/generate.py --config configs/qwen05b.yaml --adapter "$HF_USER/tool-router-qwen05b-sft" --prompts "$DATA/prompts-c16.jsonl" --out "$OUT/completions-qwen05b-sft-c16.jsonl"
!python src/generate.py --config configs/qwen15b.yaml --adapter "$HF_USER/tool-router-qwen15b-sft" --prompts "$DATA/prompts-c16.jsonl" --out "$OUT/completions-qwen15b-sft-c16.jsonl"

### The scaling grid: catalogs 40 and 80 (V1-10, the other two thirds)

Eight runs: both sizes x zero-shot and fine-tuned x catalogs 40 and 80. Catalog-80 prompts
reach ~20k tokens, so these are slower than catalog 16 — budget most of a session.

Everything is resumable: `generate.py` skips step_ids already in its `--out` file, so a run
that dies costs only the prompts it had not reached.

In [ ]:
for catalog in (40, 80):
    for cfg in ("qwen05b", "qwen15b"):
        prompts = f"{DATA}/prompts-c{catalog}.jsonl"
        for arm, adapter in (("zeroshot", None), ("sft", f"{HF_USER}/tool-router-{cfg}-sft")):
            out = f"{OUT}/completions-{cfg}-{arm}-c{catalog}.jsonl"
            print(f"===== {cfg} {arm} c{catalog} =====")
            if adapter is None:
                !python src/generate.py --config configs/{cfg}.yaml --prompts "{prompts}" --out "{out}"
            else:
                !python src/generate.py --config configs/{cfg}.yaml --adapter "{adapter}" --prompts "{prompts}" --out "{out}"

### Log-prob check for the router (V1-18)

The fine-tuned 1.5B at catalog 80 again, this time recording each token's log-prob, so the Mac
can test whether low tool-name confidence predicts wrong steps. About 70 minutes. Resumable.

In [ ]:
!python src/generate.py --config configs/qwen15b.yaml --adapter "$HF_USER/tool-router-qwen15b-sft" --prompts "$DATA/prompts-c80.jsonl" --out "$OUT/completions-qwen15b-sft-c80-logprobs.jsonl" --logprobs

### Notes

If vLLM fails with an adapter, add `--backend hf`. That path goes through transformers, so it
needs peft: run `!pip install -q peft` and then `!pip uninstall -y -q torchao`, because peft's
LoRA dispatcher raises on Kaggle's torchao 0.10.0 rather than skipping it. The K/V head
expansion that keeps long prompts off SDPA's math backend is already handled in `generate.py`.

### Send the results back

`/kaggle/working` is discarded when the session stops unless you saved a version, so push the
completions to the Hub instead. This cell still runs if an earlier generate cell failed — `!`
lines do not raise on a non-zero exit — so you get whatever finished, and the partial files
resume on the next run.

In [ ]:
from huggingface_hub import HfApi

REPO = f"{HF_USER}/tool-router-results"
api = HfApi()
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=OUT, path_in_repo="completions", repo_id=REPO, repo_type="dataset")
print("uploaded:", sorted(os.listdir(OUT)))